In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from pulp import LpProblem, LpVariable, LpMaximize, lpSum, LpStatus, value



In [2]:
BASE_DIR = Path('..') / 'data'

predictions_df = pd.read_csv(BASE_DIR / 'gold' / 'fih_predictions.csv')
coords_df = pd.read_csv(BASE_DIR / 'bronze' / 'outlet_coordinates.csv')

# Province Assignment to Outlets
PROVINCE_BBOXES = {
    'Western':      (6.70, 79.80, 7.50, 80.20),
    'Central':      (6.80, 80.20, 7.50, 81.00),
    'NorthWestern': (7.50, 79.80, 8.20, 80.40),
    'Southern':     (5.90, 80.00, 6.80, 81.20),
}

def assign_province(lat, lon):

    for province, (lat_min, lon_min, lat_max, lon_max) in PROVINCE_BBOXES.items():
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return province
    return 'Unknown'

valid_coords = coords_df[
    (coords_df['Latitude']  > 5) & (coords_df['Latitude']  < 10) &
    (coords_df['Longitude'] > 79) & (coords_df['Longitude'] < 82)
].copy()


valid_coords['Province'] = valid_coords.apply(
    lambda row: assign_province(row['Latitude'], row['Longitude']),
    axis=1
)

print("\nProvince distribution:")
print(valid_coords['Province'].value_counts())
print(f"\nUnresolved (Unknown): {(valid_coords['Province'] == 'Unknown').sum()}")




Province distribution:
Province
Western         9358
Central         4064
NorthWestern    3376
Southern        2962
Name: count, dtype: int64

Unresolved (Unknown): 0


In [3]:

outlet_master = pd.read_csv(BASE_DIR / 'silver' / 'outlet_master.csv')

# Merge: predictions + coordinates (with province) + outlet master
df = (
    predictions_df
    .merge(valid_coords[['Outlet_ID', 'Latitude', 'Longitude', 'Province']],
           on='Outlet_ID', how='inner')
    .merge(outlet_master[['Outlet_ID', 'Outlet_Type', 'Outlet_Size', 'Cooler_Count']],
           on='Outlet_ID', how='left')
)

print(f"Final merged dataset: {len(df)} outlets")
print(f"Columns: {df.columns.tolist()}")



# Filter out Western Province outlets 
western_df = df[df['Province'] == 'Western'].copy()

print(f"Western Province outlets: {len(western_df)}")


Final merged dataset: 19564 outlets
Columns: ['Outlet_ID', 'Maximum_Monthly_Liters', 'Latitude', 'Longitude', 'Province', 'Outlet_Type', 'Outlet_Size', 'Cooler_Count']
Western Province outlets: 9259


In [ ]:

#  CONFIGURATION
TOTAL_BUDGET = 5_000_000
SPEND_TIERS  = SPEND_TIERS = [0, 500, 1_000, 2_000, 5_000, 10_000]

# Outlet type responsiveness multipliers
# Grocery/Bakery/Eatery/Kiosk: high beverage relevance, respond well to promo
# Hotel/SMMT: moderate — bulk buyers but less price-elastic to trade spend
# Pharmacy: low — beverages are incidental, promo spend has minimal lift
OUTLET_TYPE_MULTIPLIER = {
    'Grocery':  1.20,
    'Bakery':   1.15,
    'Eatery':   1.15,
    'Kiosk':    1.10,
    'Hotel':    0.90,
    'SMMT':     0.85,
    'Pharmacy': 0.60,
}

#Computing historical average 
monthly_agg = pd.read_csv(BASE_DIR / 'silver' /'transactions_monthly_aggregated.csv')

monthly_avg = (
    monthly_agg
    .groupby('Outlet_ID')['monthly_volume']
    .mean()
    .reset_index()
    .rename(columns={'monthly_volume': 'avg_monthly_volume'})
)

df = western_df.merge(monthly_avg, on='Outlet_ID', how='left')
df['avg_monthly_volume'] = df['avg_monthly_volume'].fillna(0)

# Latent gap calculation
df['latent_gap']       = (df['Maximum_Monthly_Liters'] - df['avg_monthly_volume']).clip(lower=0)
df['realisation_rate'] = (
    df['avg_monthly_volume'] / df['Maximum_Monthly_Liters'].replace(0, np.nan)
).fillna(0).clip(0, 1)

# Alpha (responsiveness score) 
def minmax(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else pd.Series(0.5, index=s.index)

gap_score      = minmax(df['latent_gap'])
headroom_score = 1 - df['realisation_rate']
SIZE_MAP = {'small': 1, 'Medium': 2, 'Large': 3, 'Extra Large': 4}
df['outlet_size_numeric'] = df['Outlet_Size'].map(SIZE_MAP).fillna(1)
size_score     = minmax(df['outlet_size_numeric'])
cooler_score   = minmax(df['Cooler_Count'])

df['alpha'] = (
    0.40 * gap_score +
    0.30 * headroom_score +
    0.20 * size_score +
    0.10 * cooler_score
)

# Applying outlet type multiplier
df['type_multiplier'] = df['Outlet_Type'].map(OUTLET_TYPE_MULTIPLIER).fillna(1.0)
df['alpha'] = (df['alpha'] * df['type_multiplier']).clip(0, 1)

# zero coolers → cap alpha at 0.3 (no cold storage = hard ceiling)
df.loc[df['Cooler_Count'] == 0, 'alpha'] = df.loc[df['Cooler_Count'] == 0, 'alpha'].clip(upper=0.3)

# Pre-compute lift for every outlet × tier 
reference_spend = TOTAL_BUDGET / len(df)

def compute_lift(spend, alpha, latent_gap, ref=reference_spend):
    if spend == 0:
        return 0.0
    return alpha * np.log(1 + spend / ref) * latent_gap

records = []
for _, row in df.iterrows():
    for t, spend in enumerate(SPEND_TIERS):
        records.append({
            'Outlet_ID':    row['Outlet_ID'],
            'tier':         t,
            'spend':        spend,
            'lift':         compute_lift(spend, row['alpha'], row['latent_gap']),
        })

tier_df      = pd.DataFrame(records)
lift_lookup  = tier_df.set_index(['Outlet_ID', 'tier'])['lift'].to_dict()
spend_lookup = tier_df.set_index(['Outlet_ID', 'tier'])['spend'].to_dict()
outlets      = df['Outlet_ID'].tolist()

# Solving ILP
prob = LpProblem("MarketingSpendOptimization", LpMaximize)

x = {
    (o, t): LpVariable(f"x_{o}_{t}", cat='Binary')
    for o in outlets
    for t in range(len(SPEND_TIERS))
}

# Objective: maximize total incremental liters
prob += lpSum(
    x[(o, t)] * lift_lookup.get((o, t), 0)
    for o in outlets
    for t in range(len(SPEND_TIERS))
)

# Constraint 1: each outlet assigned exactly one tier
for o in outlets:
    prob += lpSum(x[(o, t)] for t in range(len(SPEND_TIERS))) == 1

# Constraint 2: total spend within budget
prob += lpSum(
    x[(o, t)] * spend_lookup.get((o, t), 0)
    for o in outlets
    for t in range(len(SPEND_TIERS))
) <= TOTAL_BUDGET

# Solve
prob.solve()
print(f"Solver status : {LpStatus[prob.status]}")
print(f"Optimal total lift : {value(prob.objective):,.1f} liters")

# Results 
results = []
for o in outlets:
    for t in range(len(SPEND_TIERS)):
        if value(x[(o, t)]) is not None and value(x[(o, t)]) > 0.5:
            results.append({
                'Outlet_ID':                  o,
                'Trade_Spend_Allocation_LKR': SPEND_TIERS[t],
                'assigned_tier':              t,
                'expected_lift_liters':       lift_lookup.get((o, t), 0),
            })

results_df = pd.DataFrame(results)

final = results_df.merge(
    df[['Outlet_ID', 'Outlet_Type', 'Maximum_Monthly_Liters',
        'avg_monthly_volume', 'latent_gap', 'realisation_rate', 'alpha']],
    on='Outlet_ID'
)

# Summary and save 
print(f"\nTotal spend    : LKR {final['Trade_Spend_Allocation_LKR'].sum():>12,.0f}")
print(f"Budget used    : {final['Trade_Spend_Allocation_LKR'].sum() / TOTAL_BUDGET * 100:.1f}%")
print(f"Outlets funded : {(final['Trade_Spend_Allocation_LKR'] > 0).sum()} / {len(df)}")
print(f"Total lift     : {final['expected_lift_liters'].sum():,.1f} liters\n")

print("\n── By Outlet Type ──")
print(final.groupby('Outlet_Type')[['Trade_Spend_Allocation_LKR', 'expected_lift_liters']].sum().to_string())

print("\n── By Tier ──")
print(final.groupby('assigned_tier').agg(
    outlet_count=('Outlet_ID', 'count'),
    total_spend=('Trade_Spend_Allocation_LKR', 'sum'),
    total_lift=('expected_lift_liters', 'sum')
).to_string())

submission_df = final[['Outlet_ID', 'Trade_Spend_Allocation_LKR']].copy()
submission_path = Path('..') / 'data' / 'gold' / 'fih_budget_allocations.csv'
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved: {submission_path}")
print(f"Rows: {len(submission_df)}")


Solver status : Optimal
Optimal total lift : 892,640.0 liters

Total spend    : LKR    5,000,000
Budget used    : 100.0%
Outlets funded : 3083 / 9259
Total lift     : 892,640.0 liters


── By Outlet Type ──
             Trade_Spend_Allocation_LKR  expected_lift_liters
Outlet_Type                                                  
Bakery                           894000         159283.925231
Eatery                           797500         148205.012044
Grocery                         1055500         198610.415687
Hotel                            598500         107718.344759
Kiosk                            819500         141599.919018
Pharmacy                         330500          48901.703817
SMMT                             504500          88320.644170

── By Tier ──
               outlet_count  total_spend     total_lift
assigned_tier                                          
0                      6176            0       0.000000
1                      1172       586000   63689.903

In [23]:
#Breakdown of spend tier allocation by outlet type

results_with_type = results_df.merge(
    outlet_master[['Outlet_ID', 'Outlet_Type', 'Outlet_Size']],
    on='Outlet_ID',
    how='left'
)

tier_by_type = results_with_type.groupby(
    ['Outlet_Type', 'Trade_Spend_Allocation_LKR']
).size().unstack(fill_value=0)

tier_by_type['TOTAL RECEIVING SPEND'] = (
    tier_by_type.drop(columns=0, errors='ignore').sum(axis=1)
)
tier_by_type['TOTAL OUTLETS'] = (
    results_with_type.groupby('Outlet_Type')['Outlet_ID'].count()
)

print("=== SPEND TIER ALLOCATION BY OUTLET TYPE ===")
print("(columns are spend tiers in LKR, values are number of outlets)")
print()
print(tier_by_type.to_string())
print()

#Breakdown of spend tier allocation by outlet size

tier_by_size = results_with_type.groupby(
    ['Outlet_Size', 'Trade_Spend_Allocation_LKR']
).size().unstack(fill_value=0)

tier_by_size['TOTAL RECEIVING SPEND'] = (
    tier_by_size.drop(columns=0, errors='ignore').sum(axis=1)
)
tier_by_size['TOTAL OUTLETS'] = (
    results_with_type.groupby('Outlet_Size')['Outlet_ID'].count()
)

print("=== SPEND TIER ALLOCATION BY OUTLET SIZE ===")
print()
print(tier_by_size.to_string())
print()

# Total spend committed per outlet type
print("=== TOTAL BUDGET COMMITTED BY OUTLET TYPE ===")
spend_by_type = results_with_type.groupby('Outlet_Type')['Trade_Spend_Allocation_LKR'].sum()
for outlet_type, total in spend_by_type.sort_values(ascending=False).items():
    pct = total / TOTAL_BUDGET * 100
    print(f"  {outlet_type:<15} LKR {total:>10,.0f}  ({pct:.1f}% of budget)")

=== SPEND TIER ALLOCATION BY OUTLET TYPE ===
(columns are spend tiers in LKR, values are number of outlets)

Trade_Spend_Allocation_LKR    0  500  1000  2000  5000  10000  TOTAL RECEIVING SPEND  TOTAL OUTLETS
Outlet_Type                                                                                        
Bakery                      933  194    81   153    64      9                    501           1434
Eatery                      807  233    69   136    52      8                    498           1305
Grocery                     875  225    93   190    60     17                    585           1460
Hotel                       908   87   160    75    41      4                    367           1275
Kiosk                       741  193   103   140    58      5                    499           1240
Pharmacy                    947  167    92    60     7      0                    326           1273
SMMT                        965   73   129    72    27      6                    307       

In [24]:
# Average historical volume and lift efficiency by outlet size

size_analysis = df.groupby('Outlet_Size').agg(
    outlet_count        = ('Outlet_ID', 'count'),
    avg_hist_volume     = ('avg_monthly_volume', 'mean'),
    avg_predicted       = ('Maximum_Monthly_Liters', 'mean'),
    avg_latent_gap      = ('latent_gap', 'mean'),
    avg_realisation_rate= ('realisation_rate', 'mean'),
).round(2)

# Add lift per LKR for each spend tier
for tier in SPEND_TIERS[1:]:
    df[f'lift_{tier}'] = df.apply(
        lambda r: compute_lift(tier, r['alpha'], r['latent_gap']), axis=1
    )
    size_analysis[f'Lift_per_LKR_at_{tier}'] = (
        df.groupby('Outlet_Size')[f'lift_{tier}'].mean() / tier
    ).round(6)

print("=== HISTORICAL VOLUME & LIFT EFFICIENCY BY OUTLET SIZE ===")
print("(Western Province merged dataset only)")
print()
print(size_analysis.to_string())

=== HISTORICAL VOLUME & LIFT EFFICIENCY BY OUTLET SIZE ===
(Western Province merged dataset only)

             outlet_count  avg_hist_volume  avg_predicted  avg_latent_gap  avg_realisation_rate  Lift_per_LKR_at_500  Lift_per_LKR_at_1000  Lift_per_LKR_at_2000  Lift_per_LKR_at_5000  Lift_per_LKR_at_10000
Outlet_Size                                                                                                                                                                                                  
Extra Large           446          1286.78        2017.16          730.38                  0.64             0.534397              0.427241              0.315620              0.189834               0.121139
Large                1308           533.36         950.36          417.00                  0.57             0.236601              0.189158              0.139739              0.084048               0.053633
Medium               2705           149.98         338.24          188.26    